# Análise da Camada Silver

Este notebook analisa a BigTable da camada Silver do projeto Mapa da Cidadania e Acesso à Informação no DF.

A tabela principal é `tb_mapa_cidadania_ra_ano_silver.csv`.

A granularidade da tabela é:

```text
1 linha = 1 Região Administrativa + 1 ano.
```

A base foi construída a partir da integração entre a PDAD-A 2024 e as Projeções Populacionais por Região Administrativa 2020-2030.

A base Participa DF/LAI não possui campo territorial confiável de Região Administrativa. Por isso, seus dados não foram distribuídos artificialmente por RA.

Para preservar a coerência metodológica, os indicadores da LAI foram agregados no nível Distrito Federal + ano e incorporados à BigTable como variáveis anuais de contexto, com prefixo `lai_df_`.

Dessa forma, a BigTable mantém a granularidade Região Administrativa + ano, sem produzir inferências territoriais indevidas sobre os pedidos de LAI.

## 1. Carregamento da BigTable

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("viridis")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "silver":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "Data Layer":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "Transformer":
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / "Data Layer" / "silver"

df = pd.read_csv(SILVER_DIR / "tb_mapa_cidadania_ra_ano_silver.csv")
df.head()

## 2. Diagnóstico estrutural

Verificações de linhas, colunas, tipos de dados, período disponível, quantidade de Regiões Administrativas, nulos e duplicidades.

In [ ]:
diagnostico = pd.DataFrame({
    "metrica": [
        "linhas",
        "colunas",
        "ano_minimo",
        "ano_maximo",
        "qtd_anos",
        "qtd_regioes_administrativas",
        "total_nulos",
        "duplicidades_ra_ano",
    ],
    "valor": [
        len(df),
        len(df.columns),
        int(df["ano"].min()),
        int(df["ano"].max()),
        df["ano"].nunique(),
        df["regiao_administrativa"].nunique(),
        int(df.isna().sum().sum()),
        int(df.duplicated(["regiao_administrativa", "ano"]).sum()),
    ],
})

display(diagnostico)
display(df.dtypes.rename("tipo").reset_index().rename(columns={"index": "coluna"}))
display(df.isna().sum().rename("qtd_nulos").reset_index().rename(columns={"index": "coluna"}))

## 3. Validação da chave

A chave lógica da BigTable é `regiao_administrativa + ano`. O resultado esperado é 0 duplicidades.

In [ ]:
duplicidades = (
    df.groupby(["regiao_administrativa", "ano"])
    .size()
    .reset_index(name="qtd")
    .query("qtd > 1")
)

duplicidades

## 4. Diagnóstico dos nulos socioeconômicos

Validação dos indicadores principais derivados da PDAD-A 2024. O resultado esperado é 0 linhas com nulos socioeconômicos.

In [ ]:
cols_pdad = [
    "renda_media_ponderada",
    "idade_media_ponderada",
    "media_moradores_por_domicilio",
    "percentual_baixa_escolaridade",
    "percentual_domicilios_com_internet",
    "percentual_domicilios_proprios",
    "percentual_domicilios_alugados",
]

df[df[cols_pdad].isna().any(axis=1)][
    ["ano", "regiao_administrativa"] + cols_pdad
]

## 5. Análise populacional

Análise de população total por RA, população por ano, RAs mais populosas e variação populacional entre 2023 e 2025.

In [ ]:
pop_ano = df.groupby("ano", as_index=False)["populacao_total"].sum()
pop_ra = df.groupby("regiao_administrativa", as_index=False)["populacao_total"].mean().sort_values("populacao_total", ascending=False)

pop_variacao = (
    df.pivot(index="regiao_administrativa", columns="ano", values="populacao_total")
    .reset_index()
)
pop_variacao["variacao_2023_2025"] = pop_variacao[2025] - pop_variacao[2023]
pop_variacao["variacao_percentual_2023_2025"] = 100 * pop_variacao["variacao_2023_2025"] / pop_variacao[2023]
pop_variacao = pop_variacao.sort_values("variacao_2023_2025", ascending=False)

display(pop_ano)
display(pop_ra.head(10))
display(pop_variacao.head(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=pop_ano, x="ano", y="populacao_total", marker="o", ax=axes[0])
axes[0].set_title("População total projetada por ano")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("População")

sns.barplot(data=pop_ra.head(10), y="regiao_administrativa", x="populacao_total", ax=axes[1])
axes[1].set_title("10 RAs mais populosas")
axes[1].set_xlabel("População média 2023-2025")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 6. Análise socioeconômica

Análise de renda média ponderada, idade média ponderada, baixa escolaridade, internet domiciliar, domicílios próprios e domicílios alugados.

In [ ]:
socio = (
    df.groupby("regiao_administrativa", as_index=False)[cols_pdad]
    .mean()
    .sort_values("renda_media_ponderada")
)

display(socio.head(10))
display(socio.describe())

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

sns.barplot(data=socio.head(10), y="regiao_administrativa", x="renda_media_ponderada", ax=axes[0, 0])
axes[0, 0].set_title("10 menores rendas médias ponderadas")
axes[0, 0].set_xlabel("Renda média ponderada")
axes[0, 0].set_ylabel("")

baixa = socio.sort_values("percentual_baixa_escolaridade", ascending=False).head(10)
sns.barplot(data=baixa, y="regiao_administrativa", x="percentual_baixa_escolaridade", ax=axes[0, 1])
axes[0, 1].set_title("10 maiores percentuais de baixa escolaridade")
axes[0, 1].set_xlabel("Percentual")
axes[0, 1].set_ylabel("")

internet = socio.sort_values("percentual_domicilios_com_internet").head(10)
sns.barplot(data=internet, y="regiao_administrativa", x="percentual_domicilios_com_internet", ax=axes[1, 0])
axes[1, 0].set_title("10 menores percentuais de domicílios com internet")
axes[1, 0].set_xlabel("Percentual")
axes[1, 0].set_ylabel("")

alugados = socio.sort_values("percentual_domicilios_alugados", ascending=False).head(10)
sns.barplot(data=alugados, y="regiao_administrativa", x="percentual_domicilios_alugados", ax=axes[1, 1])
axes[1, 1].set_title("10 maiores percentuais de domicílios alugados")
axes[1, 1].set_xlabel("Percentual")
axes[1, 1].set_ylabel("")

plt.tight_layout()
plt.show()

## 7. Indicadores anuais de LAI no DF

Os indicadores com prefixo `lai_df_` foram calculados no nível Distrito Federal + ano. Eles são repetidos para todas as Regiões Administrativas dentro do mesmo ano apenas como variáveis anuais de contexto.

Esses campos não representam pedidos de LAI por Região Administrativa.

In [ ]:
cols_lai_df = [
    "lai_df_qtd_pedidos_ano",
    "lai_df_qtd_recursos_ano",
    "lai_df_qtd_satisfacoes_ano",
    "lai_df_taxa_recurso_ano",
    "lai_df_tempo_medio_resposta_dias",
    "lai_df_percentual_acesso_concedido",
    "lai_df_percentual_acesso_negado",
    "lai_df_percentual_acesso_parcial",
    "lai_df_percentual_sem_resposta",
    "lai_df_qtd_orgaos_demandados",
]

lai_df_ano = df[["ano"] + cols_lai_df].drop_duplicates().sort_values("ano")
display(lai_df_ano)

variacoes_lai = df.groupby("ano")[cols_lai_df].nunique(dropna=False).reset_index()
display(variacoes_lai)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

sns.lineplot(data=lai_df_ano, x="ano", y="lai_df_qtd_pedidos_ano", marker="o", ax=axes[0, 0])
axes[0, 0].set_title("Pedidos LAI no DF por ano")
axes[0, 0].set_xlabel("Ano")
axes[0, 0].set_ylabel("Pedidos")

sns.lineplot(data=lai_df_ano, x="ano", y="lai_df_qtd_recursos_ano", marker="o", ax=axes[0, 1])
axes[0, 1].set_title("Recursos LAI no DF por ano")
axes[0, 1].set_xlabel("Ano")
axes[0, 1].set_ylabel("Recursos")

sns.lineplot(data=lai_df_ano, x="ano", y="lai_df_taxa_recurso_ano", marker="o", ax=axes[1, 0])
axes[1, 0].set_title("Taxa de recurso LAI no DF")
axes[1, 0].set_xlabel("Ano")
axes[1, 0].set_ylabel("Taxa")

sns.lineplot(data=lai_df_ano, x="ano", y="lai_df_tempo_medio_resposta_dias", marker="o", ax=axes[1, 1])
axes[1, 1].set_title("Tempo médio de resposta LAI no DF")
axes[1, 1].set_xlabel("Ano")
axes[1, 1].set_ylabel("Dias")

plt.tight_layout()
plt.show()

In [ ]:
lai_percentuais = lai_df_ano[[
    "ano",
    "lai_df_percentual_acesso_concedido",
    "lai_df_percentual_acesso_negado",
    "lai_df_percentual_acesso_parcial",
    "lai_df_percentual_sem_resposta",
]].melt(id_vars="ano", var_name="indicador", value_name="percentual")

plt.figure(figsize=(14, 6))
sns.lineplot(data=lai_percentuais, x="ano", y="percentual", hue="indicador", marker="o")
plt.title("Percentuais anuais de LAI no DF")
plt.xlabel("Ano")
plt.ylabel("Percentual")
plt.legend(title="Indicador", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8. Conclusões da Silver

A camada Silver passou a conter uma BigTable territorial consolidada por Região Administrativa e ano, com 99 linhas esperadas para 33 Regiões Administrativas nos anos 2023, 2024 e 2025.

Principais achados:

- A tabela possui uma linha por `regiao_administrativa + ano`.
- Não há duplicidades na chave lógica.
- Os indicadores socioeconômicos principais não possuem nulos.
- As projeções populacionais permitem acompanhar variações por RA entre 2023 e 2025.
- Os indicadores PDAD-A 2024 funcionam como perfil socioeconômico de referência para os três anos analisados.

Limitações:

- A PDAD-A 2024 é usada como fotografia socioeconômica de referência, não como série anual.
- A BigTable territorial contém indicadores LAI anuais do DF como contexto, mas não contém indicadores LAI específicos por RA.

A base Participa DF/LAI não possui campo territorial confiável de Região Administrativa. Por isso, seus dados não foram distribuídos artificialmente por RA.

Para preservar a coerência metodológica, os indicadores da LAI foram agregados no nível Distrito Federal + ano e incorporados à BigTable como variáveis anuais de contexto, com prefixo `lai_df_`.

Dessa forma, a BigTable mantém a granularidade Região Administrativa + ano, sem produzir inferências territoriais indevidas sobre os pedidos de LAI.

Próximos passos para Gold:

- Criar dimensões de Região Administrativa e tempo.
- Separar fatos populacionais e socioeconômicos conforme a modelagem dimensional.
- Evoluir a BigTable para os indicadores analíticos finais do projeto.